# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 20 · Relationship evidence and goal-frame geometry

**One bounded diagnostic milestone: zero model fits, zero new optimizer steps.**

Round 8 completed successfully but its feature gate failed. We first inspect the actual saved evidence, then verify existing artifacts and build six observed-only goal-frame channels on 32 training plays. These channels are not integrated into a model and have no measured accuracy benefit.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round9')
OUT = Path('/home/sagemaker-user/nfl-feature-round9-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open the existing NFL space and extract Round 9 first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage):
    process = subprocess.Popen([str(PY), str(KIT/'run_round.py'), stage],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(2)
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Preserve inputs and export the report; do not train or change settings.')
def show(fig, name):
    visuals.save(fig, OUT, name).show()


## Existing Round 8 evidence

These figures are real user-provided aggregate results. They are not a new run. The paired interval crosses zero and the history model is worse than the saved tree. Nearly identical objectives motivate a signal audit; they do not prove a wiring bug.

In [ ]:
show(visuals.round8_scores(KIT), 'round8_scores')
show(visuals.round8_objective_gap(KIT), 'round8_objective_gap')
show(visuals.round8_horizons(KIT), 'round8_horizon_concentration')

## Verify the completed parent experiment

Require `signal_preflight_passed`. Only small manifests and source/checkpoint/input hashes are checked. No old scientific runner or evaluation input array is opened.

In [ ]:
run('preflight')
print(json.dumps(json.loads((OUT/'preflight.json').read_text()), indent=2))

## New representation smoke: goal-aligned pair geometry

Require `goal_frame_smoke_passed`. Six numeric channels describe separation/relative velocity in the source-player-to-landing frame and peer versus source approach to that landmark. Four masks partition known roles/sides, not coverage assignments. There is no interpolation or future-coordinate feature input.

In [ ]:
run('feature-smoke')

In [ ]:
show(visuals.feature_support(OUT), 'goal_frame_support')
show(visuals.feature_range(OUT), 'goal_frame_magnitude')

## Checkpoint

Save this notebook. Continue to notebook 21 only after both statuses passed. No current or future validation labels are used to select these plays or construct these features. Magnitude/support plots are engineering checks, not feature importance.